In [1]:
import os
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
#import rasterio
#from rasterio.mask import mask

In [2]:
project_root = os.path.dirname(os.path.dirname("test.ipynb"))
data_dir = os.path.join(project_root, "data")
raster_dir = os.path.join(data_dir, "rasters")
transient_dir = os.path.join(project_root, "transients")
output_dir = os.path.join(project_root, "outputs")

energy_raster_path = os.path.join(raster_dir, "GHS_BUILT_S_timeseries_points.gpkg")


shapefile_path = os.path.join(data_dir, "ne_10m_admin_0_countries.shp")  # may need other files rather than just shp?
raster_path = os.path.join(data_dir, "gpw_v4_population_density_rev11_2020_30_min.tif")
country_energy_path = os.path.join(data_dir, "Country Energy Data.xlsx")

In [167]:
energy_timeseries = gpd.read_file(energy_raster_path)

led_data = pd.read_excel(country_energy_path)
chosen_column = [str(col) for col in led_data.columns if "Chosen" in str(col)][0]
led_data = led_data[led_data[chosen_column] > 0].dropna(subset=[chosen_column])


year = 2025
place_ocean = True
all_leds_gdf = gpd.GeoDataFrame()

In [164]:
energy_timeseries

,point_index,country,1975,1980,1985,1990,1995,2000,2005,2010,2015,2020,2025,geometry
0,16058,Greenland,2714,3174,3633,4131,4131,4131,4329,4534,4745,4982,4982,POINT (-70.75125 77.84958)
1,16228,Norway,345,395,445,500,500,500,516,537,560,584,584,POINT (14.24875 77.84958)
2,16233,None,20748,24243,27745,31299,31299,31299,32818,34340,35916,37650,37650,POINT (16.74875 77.84958)
3,16408,None,8140,9347,10552,11855,11855,11855,12384,12924,13483,14135,14135,POINT (104.24875 77.84958)
4,16781,None,30580,35969,41378,46913,46913,46913,49241,51583,53981,56628,56628,POINT (-69.25125 77.34958)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52180,208302,None,3295,3646,3998,4387,4433,4469,4485,4501,4531,4607,4607,POINT (-68.75125 -55.65042)
52181,208303,None,1529,1702,1862,2034,2059,2073,2085,2098,2111,2156,2156,POINT (-68.25125 -55.65042)
52182,208304,None,1797,2166,2554,2939,2969,2987,3001,3015,3039,3090,3090,POINT (-67.75125 -55.65042)
52183,208305,None,1760,1940,2119,2311,2334,2354,2358,2363,2372,2419,2419,POINT (-67.25125 -55.65042)


In [169]:
all_leds_gdf[all_leds_gdf['Country']=="South Korea"]

,geometry,Country,Raster_Density
2234,POINT (126.74875 37.34958),South Korea,291082140
2235,POINT (127.24875 37.34958),South Korea,245595838
2236,POINT (127.24875 36.84958),South Korea,195166464
2237,POINT (128.74875 35.34958),South Korea,152364932
2238,POINT (129.24875 35.34958),South Korea,151058050
...,...,...,...
2300,POINT (128.74875 1.84958),South Korea,46530
2301,POINT (126.74875 -7.65042),South Korea,133549
2302,POINT (127.24875 34.34958),South Korea,12652149
2303,POINT (127.74875 39.84958),South Korea,19393741


In [168]:
for index, row in led_data.iterrows():

    country_name = row['Entity']
    num_leds = int(row['Round'])
    leds_placed = 0

    values_array = energy_timeseries[energy_timeseries['country'] == country_name]
    values_array = values_array[["point_index", f"{year}", "geometry"]].sort_values(f"{year}", ascending=False)

    available_cells = len(values_array)
    missing_leds = num_leds - available_cells
    
    if available_cells == 0:
        print(f"Could not find {country_name} in raster data, skipping...")

    else:

        for leds in range(0,num_leds-leds_placed): # Place LEDs on the land-space

            if leds_placed < available_cells: 
                
                all_leds_gdf = pd.concat([all_leds_gdf, gpd.GeoDataFrame({'geometry': [values_array["geometry"].iloc[leds]],
                                                                        'Country': [country_name],
                                                                        'Raster_Density': [values_array[f"{year}"].iloc[leds]]
                                                                        }, geometry='geometry')], ignore_index=True)
                leds_placed += 1

            else:

                if place_ocean == True:
                    
                    if leds_placed >= num_leds:
                        break

                    print(f"Warning: {country_name} requested {num_leds} LEDs, but only {available_cells} cells are available. Attempting to place {missing_leds} LEDs in surrounding area.")
                    values_sorted = energy_timeseries.iloc[energy_timeseries.geometry.x.argsort().values].reset_index(drop=True)
                    surround = 1
                    
                    while leds_placed < num_leds:

                        values_sorted_filtered = values_sorted[values_sorted['country'] == country_name]
                        surround_indices = [x-surround for x in values_sorted_filtered.index] + [x+surround for x in values_sorted_filtered.index] 
                        surround_indices = values_sorted.loc[surround_indices].query("country.isnull()").index
                        values_sorted.loc[surround_indices, "country"] = country_name

                        for leds in surround_indices: # Place remaining LEDs on the land-space
                            all_leds_gdf = pd.concat([all_leds_gdf, gpd.GeoDataFrame({'geometry': [values_sorted["geometry"].iloc[leds]],
                                                                                    'Country': [country_name],
                                                                                    'Raster_Density': [values_sorted[f"{year}"].iloc[leds]]
                                                                                    }, geometry='geometry')], ignore_index=True)
                            leds_placed += 1
                            if leds_placed >= num_leds:
                                break
                        surround += 1
                
                else: 
                    print(f"Warning: {country_name} requested {num_leds} LEDs, but only {leds_placed} were placed due to not having enough space.")
                    break

Could not find Hong Kong S.A.R. in raster data, skipping...
Could not find Bahrain in raster data, skipping...
Could not find Republic of Serbia in raster data, skipping...
Could not find Dominican Republic in raster data, skipping...
Could not find Bosnia and Herzegovina in raster data, skipping...
Could not find Ivory Coast in raster data, skipping...
Could not find United Republic of Tanzania in raster data, skipping...
Could not find Netherlands Antilles in raster data, skipping...
